# Tutorial for the WfcParser class

This tutorial describes the `WfcParser` class, which reads the Kohn-Sham wavefunctions computed by QuantumESPRESSO (the
`<prefix>.save` folder) or stored by Yambo in its `SAVE` folder. The wavefunctions are expanded in plane waves,

$$\psi_{n\mathbf{k}}(\mathbf{r}) = \frac{1}{\sqrt{\Omega}}\sum_\mathbf{G} c_n(\mathbf{G})\,e^{i(\mathbf{k}+\mathbf{G})\cdot\mathbf{r}} ,$$

and the class gives the coefficients $c_n(\mathbf{G})$ of the chosen bands at a k point, with the G vectors identified by their
Miller indices. The class computes the scalar products between states, also of different computations, and applies the symmetry
operations to the states. The k points and the bands are numbered from 1, as in QuantumESPRESSO and Yambo. Only computations
without spin are supported, and the scalar products are correct for norm-conserving pseudopotentials.

We use the results of other tutorials: the nscf computation of silicon of the QeCalculator tutorial, the SAVE folder built
from it in the Yambo tutorials, and the scf computation of graphene of the BandStructure tutorial.

In [1]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

import datetime
import numpy as np
from mppi.Parsers import WfcParser

print('Notebook executed on', datetime.datetime.now().strftime('%d/%m/%Y %H:%M'))

Notebook executed on 14/11/2026 16:09


## Wavefunctions of QuantumESPRESSO

The class is built from the `<prefix>.save` folder with the `from_qe` method. The global data (lattice, atoms, k points,
energies) are read from the `data-file-schema.xml` file, the coefficients only when they are requested

In [2]:
qe = WfcParser.from_qe('QeCalculator_tutorial/out_nscf/si_scf.save')

Wavefunctions of QeCalculator_tutorial/out_nscf/si_scf.save (QuantumESPRESSO)
number of k points 32 - number of bands 8 - number of electrons 8.0
atoms: Si [-1.2875  1.2875  1.2875], Si [ 1.2875 -1.2875 -1.2875] (bohr)


The lattice vectors and the atomic positions are in bohr, the k points in cartesian coordinates (bohr$^{-1}$) and in crystal
coordinates, the energies in eV

In [3]:
print('lattice vectors (rows):\n', qe.lattice)
print('k point 2, cartesian:', np.round(qe.kpoints[1], 6), 'crystal:', np.round(qe.kpoints_crystal[1], 6))
print('energies at k point 2:', qe.evals[1])

lattice vectors (rows):
 [[-5.15  0.    5.15]
 [ 0.    5.15  5.15]
 [-5.15  5.15  0.  ]]
k point 2, cartesian: [-0.10167  0.10167 -0.10167] crystal: [-0.       -0.        0.166667]
energies at k point 2: [-5.4127677   3.74503329  5.67122426  5.6712469   8.42096889  9.32243335
  9.32246108 11.41480272]


The `get_states` method reads the coefficients of the chosen bands at a k point. The states are a dictionary with the k point,
the reciprocal lattice vectors `b` (rows), the Miller indices of the G vectors and the coefficients, with shape (bands, G vectors)

In [4]:
states = qe.get_states(2, bands=[1, 2, 3, 4])
print(states.keys())
print('number of G vectors:', len(states['mill']), '- shape of the coefficients:', states['c'].shape)
print('first G vectors (Miller indices):\n', states['mill'][:4])

dict_keys(['k', 'b', 'mill', 'bands', 'c', 'ik'])
number of G vectors: 761 - shape of the coefficients: (4, 761)
first G vectors (Miller indices):
 [[ 0  0  0]
 [-1 -1 -1]
 [-1  0  0]
 [ 0 -1  0]]


The `overlap` method computes the scalar products $\langle\psi_m|\psi_n\rangle = \sum_\mathbf{G}c^*_m(\mathbf{G})c_n(\mathbf{G})$.
The states of a k point are orthonormal

In [5]:
states = qe.get_states(2)
O = qe.overlap(states, states)
print(np.round(np.abs(O), 6))
print('max |<m|n> - delta_mn| = %.1e' % np.abs(O - np.eye(len(O))).max())

[[1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]]
max |<m|n> - delta_mn| = 2.4e-15


## Wavefunctions of Yambo

The class is built from the `SAVE` folder with the `from_yambo` method: the global data are read from the `ns.db1` database, the
coefficients from the `ns.wf` database and its fragments. This SAVE was built from the QuantumESPRESSO computation used above, so it
contains the same k points, energies and states. Yambo stores the coefficients in single precision

In [6]:
ya = WfcParser.from_yambo('Yambo_tutorial/SAVE')
print('same k points:', np.allclose(ya.kpoints, qe.kpoints, atol=1e-5))
print('max difference of the energies: %.1e eV' % np.abs(ya.evals - qe.evals).max())

Wavefunctions of Yambo_tutorial/SAVE (Yambo)
number of k points 32 - number of bands 8 - number of electrons 8.0
atoms: Si [-1.2875  1.2875  1.2875], Si [ 1.2875 -1.2875 -1.2875] (bohr)
same k points: True
max difference of the energies: 1.3e-06 eV


The `overlap` method also works between states of two objects, provided that the k point and the cell are the same: the G
vectors are matched through their Miller indices. The scalar products between the states of QuantumESPRESSO and those of Yambo
give the identity matrix within the single precision at all the k points (the states have also the same phases)

In [7]:
err = 0.
for ik in range(1, qe.nkpoints + 1):
    O = qe.overlap(qe.get_states(ik), ya.get_states(ik))
    err = max(err, np.abs(O - np.eye(len(O))).max())
print('max |<qe_m|yambo_n> - delta_mn| over the k points = %.1e' % err)

max |<qe_m|yambo_n> - delta_mn| over the k points = 3.7e-08


## Symmetry transformations

The `transform` method applies a symmetry operation to the states, $\psi'(\mathbf{r}) = \psi(R^{-1}(\mathbf{r}-\boldsymbol{\tau}))$,
with a rotation $R$ (cartesian matrix) and a translation $\boldsymbol{\tau}$ (bohr), and possibly the time reversal (complex conjugate).
We use the graphene scf computation of the BandStructure tutorial, on a 12x12 grid that contains the K point, where the
$\pi$ and $\pi^*$ bands (4 and 5) are degenerate

In [8]:
gr = WfcParser.from_qe('BandStructure_tutorial/out_graphene/graphene.save')
ik = int(np.argmin(gr.evals[:, 4] - gr.evals[:, 3])) + 1
print('K point: index', ik, '- crystal coordinates', np.round(gr.kpoints_crystal[ik-1], 4))
print('energies (eV):', np.round(gr.evals[ik-1], 3))

Wavefunctions of BandStructure_tutorial/out_graphene/graphene.save (QuantumESPRESSO)
number of k points 19 - number of bands 8 - number of electrons 8.0
atoms: C [0. 0. 0.], C [1.3464 2.332  0.    ] (bohr)
K point: index 19 - crystal coordinates [ 0.3333 -0.3333  0.    ]
energies (eV): [-14.296 -14.296 -12.278  -1.734  -1.734   8.911  11.135  11.135]


The degenerate states at K must be mapped onto themselves by the symmetries of the crystal that leave K invariant (or map it
onto an equivalent point, $\mathbf{k}+\mathbf{G}_0$, which the `overlap` method takes into account). For each group of degenerate
states the sum of $|\langle\psi_m|\psi'_n\rangle|^2$ over the group must be equal to the number of states of the group. We check the
rotations by $2\pi/3$ about each of the two atoms and the inversion about the center of the bond, which maps K onto -K, combined
with the time reversal. The rotation about the center of the bond is not a symmetry of graphene, and it does not satisfy the
check. The last group of bands (7 and 8) is excluded, since it can be coupled with the bands not computed

In [9]:
rA, rB = gr.atoms[0][1], gr.atoms[1][1]
th = 2*np.pi/3
C3 = np.array([[np.cos(th), -np.sin(th), 0.], [np.sin(th), np.cos(th), 0.], [0., 0., 1.]])
operations = {
    'C3 about the atom A': dict(rotation=C3, translation=rA - C3 @ rA),
    'C3 about the atom B': dict(rotation=C3, translation=rB - C3 @ rB),
    'inversion and time reversal': dict(rotation=-np.eye(3), translation=rA + rB, time_reversal=True),
    'C3 about the center of the bond': dict(rotation=C3, translation=(rA + rB)/2 - C3 @ (rA + rB)/2)}

groups = [[1, 2], [3], [4, 5], [6]]
states = gr.get_states(ik, bands=range(1, 7))
for name, op in operations.items():
    O = gr.overlap(states, gr.transform(states, **op))
    sums = [(np.abs(O[np.ix_(np.array(g)-1, np.array(g)-1)])**2).sum() for g in groups]
    print('%-32s' % name, '  '.join('bands %s: %.6f' % (g, s) for g, s in zip(groups, sums)))

C3 about the atom A              bands [1, 2]: 2.000000  bands [3]: 1.000000  bands [4, 5]: 2.000000  bands [6]: 1.000000
C3 about the atom B              bands [1, 2]: 2.000000  bands [3]: 1.000000  bands [4, 5]: 2.000000  bands [6]: 1.000000
inversion and time reversal      bands [1, 2]: 2.000000  bands [3]: 1.000000  bands [4, 5]: 2.000000  bands [6]: 1.000000
C3 about the center of the bond  bands [1, 2]: 1.051670  bands [3]: 0.004427  bands [4, 5]: 0.840520  bands [6]: 0.102122
